# Manipuri ASR Transcription

This notebook transcribes Manipuri audio using the AI4Bharat IndicConformer model, processing only the first minute of audio.


In [ ]:
import torch
import nemo.collections.asr as nemo_asr
import librosa
import soundfile as sf
import os
import tempfile


In [ ]:
# Configuration
audio_file = '~/Downloads/manipuri_test.wav'
audio_file = os.path.expanduser(audio_file)  # Expand ~ to home directory
max_duration_seconds = 60  # Only transcribe first minute


In [ ]:
# Load and trim audio to first minute
print(f"Loading audio from: {audio_file}")
audio, sr = librosa.load(audio_file, sr=16000, mono=True)

# Calculate duration
duration = len(audio) / sr
print(f"Original audio duration: {duration:.2f} seconds")

# Trim to first minute (60 seconds)
max_samples = int(max_duration_seconds * sr)
if len(audio) > max_samples:
    audio_trimmed = audio[:max_samples]
    print(f"Trimming audio to first {max_duration_seconds} seconds")
else:
    audio_trimmed = audio
    print(f"Audio is shorter than {max_duration_seconds} seconds, using full audio")

# Save trimmed audio to temporary file
with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp_file:
    tmp_audio_path = tmp_file.name
    sf.write(tmp_audio_path, audio_trimmed, sr)
    print(f"Saved trimmed audio to temporary file: {tmp_audio_path}")


In [ ]:
# Load the ASR model
print("Loading ASR model...")
model = nemo_asr.models.ASRModel.from_pretrained("ai4bharat/indicconformer_stt_mni_hybrid_rnnt_large")

# Set up device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Freeze model for inference
model.freeze()
model = model.to(device)


In [ ]:
# Transcribe using CTC decoder
print("Transcribing audio...")
model.cur_decoder = "ctc"
ctc_text = model.transcribe(
    [tmp_audio_path], 
    batch_size=1,
    logprobs=False, 
    language_id='mni'
)[0]

print("\n" + "="*50)
print("TRANSCRIPTION RESULT:")
print("="*50)
print(ctc_text)
print("="*50)


In [ ]:
# Clean up temporary file
if os.path.exists(tmp_audio_path):
    os.remove(tmp_audio_path)
    print("Cleaned up temporary audio file")
